In [30]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (accuracy_score, confusion_matrix,classification_report)

In [31]:
df=pd.read_csv("/Users/nidagallusandeep/Downloads/naive_bayes_email_classification_7000.csv")

In [32]:
df.head()

,Word_Frequency,Link_Count,Capital_Ratio,Exclamation_Count,Email_Length,Attachment_Count,Unknown_Sender,Email_Type
0,52,11,0.575,11,119,2,0,Spam
1,93,17,0.398,15,284,1,1,Spam
2,15,18,0.364,9,769,3,1,Not Spam
3,72,14,0.012,3,769,4,0,Not Spam
4,61,7,0.249,14,68,4,0,Not Spam


In [33]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7000 entries, 0 to 6999
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Word_Frequency     7000 non-null   int64  
 1   Link_Count         7000 non-null   int64  
 2   Capital_Ratio      7000 non-null   float64
 3   Exclamation_Count  7000 non-null   int64  
 4   Email_Length       7000 non-null   int64  
 5   Attachment_Count   7000 non-null   int64  
 6   Unknown_Sender     7000 non-null   int64  
 7   Email_Type         7000 non-null   object 
dtypes: float64(1), int64(6), object(1)
memory usage: 437.6+ KB


In [34]:
df.shape

(7000, 8)

In [35]:
df.columns

Index(['Word_Frequency', 'Link_Count', 'Capital_Ratio', 'Exclamation_Count',
       'Email_Length', 'Attachment_Count', 'Unknown_Sender', 'Email_Type'],
      dtype='object')

In [36]:
df.describe()

,Word_Frequency,Link_Count,Capital_Ratio,Exclamation_Count,Email_Length,Attachment_Count,Unknown_Sender
count,7000.000000,7000.00000,7000.000000,7000.000000,7000.000000,7000.000000,7000.000000
mean,50.113714,10.04800,0.501165,7.510286,511.113143,2.526286,0.507571
std,28.812710,6.02872,0.286044,4.640209,282.751727,1.703361,0.499978
min,1.000000,0.00000,0.000000,0.000000,20.000000,0.000000,0.000000
25%,25.000000,5.00000,0.257000,3.000000,268.750000,1.000000,0.000000
50%,50.000000,10.00000,0.500500,7.000000,512.000000,3.000000,1.000000
75%,75.000000,15.00000,0.743000,12.000000,756.000000,4.000000,1.000000
max,100.000000,20.00000,1.000000,15.000000,1000.000000,5.000000,1.000000


In [37]:
df.isnull().sum()

Word_Frequency       0
Link_Count           0
Capital_Ratio        0
Exclamation_Count    0
Email_Length         0
Attachment_Count     0
Unknown_Sender       0
Email_Type           0
dtype: int64

In [38]:
df.duplicated().sum()

np.int64(0)

In [39]:
df["Email_Type"].value_counts()

Email_Type
Spam        3500
Not Spam    3500
Name: count, dtype: int64

# Separate Features and Target

In [40]:
X=df.drop("Email_Type",axis=1)
y=df["Email_Type"]

In [41]:
y=y.map({
    "Not Spam":0,
    "Spam":1})

In [42]:
print(y.head())

0    1
1    1
2    0
3    0
4    0
Name: Email_Type, dtype: int64


# Train-Test Split

In [43]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,train_size=0.2,random_state=42)
print(X_train.shape)
print(X_test.shape)
print(y_test.shape)
print(y_train.shape)

(1400, 7)
(5600, 7)
(5600,)
(1400,)


# from sklearn.feature_selection import SelectKBest, chi2

In [45]:
from sklearn.feature_selection import SelectKBest, chi2
selector = SelectKBest(score_func=chi2,k=5)
X_train_selected = selector.fit_transform(X_train,y_train)
X_test_selected = selector.transform(X_test)

# Display Selected Features

In [46]:
selected_features = X.columns[
    selector.get_support()]
print("\nSelected Features:")
print(selected_features.tolist())



Selected Features:
['Word_Frequency', 'Link_Count', 'Exclamation_Count', 'Email_Length', 'Attachment_Count']


# Feature Scores

In [48]:
feature_scores = pd.DataFrame({"Feature": X.columns,"Score": selector.scores_})
print("\nFeature Scores:")
print(feature_scores)


Feature Scores:
             Feature        Score
0     Word_Frequency  2309.386121
1         Link_Count   438.478503
2      Capital_Ratio     8.047090
3  Exclamation_Count   340.060505
4       Email_Length  1682.815338
5   Attachment_Count    76.535293
6     Unknown_Sender    34.153236


# Naive Bayes Model

In [50]:
from sklearn.naive_bayes import GaussianNB
model = GaussianNB()
model.fit(X_train_selected,y_train)

,"priors priors: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None
,"var_smoothing var_smoothing: float, default=1e-9Portion of the largest variance of all features that is added tovariances for calculation stability... versionadded:: 0.20",1e-09
Name,Type,Value
"class_count_ class_count_: ndarray of shape (n_classes,)number of training samples observed in each class.","ndarray[float64](2,)","[723.,677.]"
"class_prior_ class_prior_: ndarray of shape (n_classes,)probability of each class.","ndarray[float64](2,)","[0.52,0.48]"
"classes_ classes_: ndarray of shape (n_classes,)class labels known to the classifier.","ndarray[int64](2,)","[0,1]"
epsilon_ epsilon_: floatabsolute additive value to variances.,float64,7.898e-05
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,5
"theta_ theta_: ndarray of shape (n_classes, n_features)mean of each feature per class.","ndarray[float64](2, 5)","[[ 40.96, 8.49, 6.12,485.83, 2.17], [ 59.08, 12.06, 8.81,535.36, 2.91]]"
"var_ var_: ndarray of shape (n_classes, n_features)Variance of each feature per class... versionadded:: 1.0","ndarray[float64](2, 5)","[[ 750.75, 33.45, 20.29,80441. , 2.78], [ 756.26, 31.26, 20.17,76158.69, 2.7 ]]"


#  Prediction

In [51]:
y_pred=model.predict(X_test_selected)

# Accuracy

In [53]:
from sklearn.metrics import accuracy_score
accuracy=accuracy_score(y_test,y_pred)
print("Accuracy:",accuracy)

Accuracy: 0.7680357142857143


In [54]:
print("\nAccuracy %:")
print(accuracy * 100)


Accuracy %:
76.80357142857143


# Confusion Matrix

In [57]:
cm=confusion_matrix(y_test,y_pred)
print("Confusion Matrix:",cm)

Confusion Matrix: [[2223  554]
 [ 745 2078]]


# Classification Report

In [59]:
cr=classification_report(y_test,y_pred)
print("Classification Report:",cr)

Classification Report:               precision    recall  f1-score   support

           0       0.75      0.80      0.77      2777
           1       0.79      0.74      0.76      2823

    accuracy                           0.77      5600
   macro avg       0.77      0.77      0.77      5600
weighted avg       0.77      0.77      0.77      5600

